# Latency SLA Prediction

## Learning Objectives
1. Understand latency prediction as request-level feature modeling
2. Implement single-feature and multi-feature latency predictors
3. Design SLA-aware routing and batching strategies
4. Monitor and maintain SLA compliance in production

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Dict, Tuple, Optional
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Level 1: Basic SLA Prediction

In [ ]:
# Level 1: Numpy linear regression for latency prediction
# Features: batch_size, seq_len -> predicted latency (ms)
# Physical model: latency ~ base_cost + prefill_coefficient * seq_len + batch_overhead * batch_size

np.random.seed(42)

def generate_latency_dataset(
    n_samples: int = 500, noise_std: float = 8.0
) -> tuple:
    """
    Generate synthetic LLM inference latency measurements.
    True model: lat = 5 + 0.06*seq_len + 3.5*batch_size + noise
    """
    batch_sizes = np.random.choice([1, 2, 4, 8, 16, 32], size=n_samples)
    seq_lens    = np.random.randint(32, 512, size=n_samples)
    # True latency (ms) with physical relationships
    true_lat = (5.0
                + 0.06  * seq_lens
                + 3.5   * batch_sizes
                + noise_std * np.random.randn(n_samples))
    true_lat = np.maximum(true_lat, 2.0)  # floor at 2ms
    X = np.column_stack([seq_lens, batch_sizes])
    return X, true_lat, seq_lens, batch_sizes

X_train, y_train, seq_lens_tr, batch_sizes_tr = generate_latency_dataset(500)
X_test,  y_test,  seq_lens_te, batch_sizes_te = generate_latency_dataset(100, noise_std=5.0)

# Fit linear regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f'Linear Regression Latency Predictor:')
print(f'  Intercept (base cost):   {lr_model.intercept_:.2f}ms')
print(f'  seq_len coefficient:     {lr_model.coef_[0]:.4f} ms/token')
print(f'  batch_size coefficient:  {lr_model.coef_[1]:.4f} ms/request')
print(f'  MAE:  {mae:.2f}ms')
print(f'  R^2:  {r2:.4f}')

# SLA compliance at 50ms threshold
SLA_MS = 50.0
predicted_compliant = (y_pred <= SLA_MS)
actual_compliant    = (y_test <= SLA_MS)
precision  = (predicted_compliant & actual_compliant).sum() / predicted_compliant.sum()
recall     = (predicted_compliant & actual_compliant).sum() / actual_compliant.sum()
print(f'\nSLA compliance prediction (threshold={SLA_MS}ms):')
print(f'  Precision: {precision:.3f} (of predicted-compliant, {precision:.0%} truly are)')
print(f'  Recall:    {recall:.3f}   (of truly compliant, {recall:.0%} detected)')

# Show predictions for representative request types
test_cases = [(64, 1), (128, 8), (256, 16), (512, 32)]
print('\nPrediction table for representative request profiles:')
print(f'{"seq_len":>8} {"batch_sz":>9} {"predicted_ms":>14} {"SLA_met?":>10}')
for sl, bs in test_cases:
    pred_lat = lr_model.predict([[sl, bs]])[0]
    sla_met = "Yes" if pred_lat <= SLA_MS else "No"
    print(f'{sl:>8} {bs:>9} {pred_lat:>14.1f} {sla_met:>10}')

# Residual analysis: are errors correlated with features?
residuals = y_test - y_pred
corr_seqlen   = np.corrcoef(seq_lens_te, residuals)[0, 1]
corr_batchsz  = np.corrcoef(batch_sizes_te, residuals)[0, 1]
print(f'\nResidual correlation with seq_len:   {corr_seqlen:.3f}')
print(f'Residual correlation with batch_size: {corr_batchsz:.3f}')
print('(Near-zero = linear model captures the relationship well)')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(y_test, y_pred, alpha=0.5, s=30)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Latency (ms)')
axes[0].set_ylabel('Predicted Latency (ms)')
axes[0].set_title(f'Predicted vs Actual (MAE={mae:.1f}ms)')
axes[0].grid(alpha=0.3)

axes[1].scatter(y_pred, residuals, alpha=0.5, s=30, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Latency (ms)')
axes[1].set_ylabel('Residual (ms)')
axes[1].set_title('Residual Plot')
axes[1].grid(alpha=0.3)

axes[2].scatter(seq_lens_te, y_test, alpha=0.4, s=30, c=batch_sizes_te, cmap='viridis')
X_range = np.column_stack([np.linspace(32, 512, 100), np.ones(100) * 8])
axes[2].plot(np.linspace(32, 512, 100), lr_model.predict(X_range),
             'r-', linewidth=2, label='Pred (batch=8)')
axes[2].axhline(SLA_MS, color='green', linestyle='--', label=f'SLA {SLA_MS}ms')
axes[2].set_xlabel('Sequence Length')
axes[2].set_ylabel('Latency (ms)')
axes[2].set_title('Latency vs Seq Length (color=batch_size)')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Level 1 complete: linear latency predictor with SLA compliance')


## Level 2: Advanced SLA Prediction with Multiple Features

In [ ]:
# Level 2: MLP latency predictor with calibration and uncertainty estimation
# Features: seq_len, batch_size, model_params_B, is_fp16, kv_cache_tokens
# Outputs: predicted latency + confidence interval via dropout-based uncertainty

class LatencyPredictorMLP(nn.Module):
    """
    3-layer MLP latency predictor.
    Dropout during inference enables MC-dropout uncertainty estimation.
    """
    def __init__(self, n_features: int = 5, hidden_dim: int = 64, dropout: float = 0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

    def predict_with_uncertainty(
        self, x: torch.Tensor, n_samples: int = 30
    ) -> tuple:
        """
        MC-dropout: run n_samples forward passes (each with different dropout mask).
        Returns (mean_prediction, std_prediction) as uncertainty estimate.
        """
        self.train()  # Enable dropout during inference
        preds = []
        with torch.no_grad():
            for _ in range(n_samples):
                preds.append(self(x))
        self.eval()
        preds_t = torch.stack(preds, dim=0)   # [n_samples, batch]
        return preds_t.mean(dim=0), preds_t.std(dim=0)

def generate_multifeature_latency_data(n: int = 800) -> tuple:
    """Generate synthetic LLM latency data with 5 features."""
    np.random.seed(42)
    seq_len     = np.random.randint(32, 2048, n).astype(float)
    batch_size  = np.random.choice([1, 2, 4, 8, 16, 32], n).astype(float)
    model_B     = np.random.choice([0.5, 3.0, 7.0, 13.0, 70.0], n)
    is_fp16     = np.random.randint(0, 2, n).astype(float)
    kv_tokens   = np.random.randint(0, 4096, n).astype(float)

    # True latency model (ms):
    # base + prefill_coeff*seq + decode_coeff*model_B + batch_overhead + kv_penalty
    lat = (10
           + 0.04  * seq_len
           + 5.0   * model_B
           + 2.0   * batch_size
           + 0.005 * kv_tokens
           - 8.0   * is_fp16   # fp16 saves ~8ms
           + 15    * np.random.randn(n))
    lat = np.maximum(lat, 5.0)

    X = np.column_stack([seq_len, batch_size, model_B, is_fp16, kv_tokens])
    # Normalize
    X_mean, X_std = X.mean(axis=0), X.std(axis=0)
    X_norm = (X - X_mean) / (X_std + 1e-9)
    return X_norm, lat, X_mean, X_std

X_all, y_all, feat_mean, feat_std = generate_multifeature_latency_data(800)
split = 640
X_tr = torch.tensor(X_all[:split], dtype=torch.float32, device=device)
y_tr = torch.tensor(y_all[:split], dtype=torch.float32, device=device)
X_val = torch.tensor(X_all[split:], dtype=torch.float32, device=device)
y_val = torch.tensor(y_all[split:], dtype=torch.float32, device=device)

torch.manual_seed(42)
mlp = LatencyPredictorMLP(n_features=5, hidden_dim=64).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=5e-3, weight_decay=1e-4)
loss_fn = nn.HuberLoss(delta=15.0)  # Huber: robust to outlier latencies

batch_size_train = 64
n_epochs = 60
train_losses, val_losses = [], []

mlp.train()
for epoch in range(n_epochs):
    perm = torch.randperm(len(X_tr), device=device)
    epoch_loss = 0.0
    for start in range(0, len(X_tr), batch_size_train):
        idx = perm[start:start + batch_size_train]
        xb, yb = X_tr[idx], y_tr[idx]
        try:
            pred = mlp(xb)
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("OOM: reduce batch_size_train")
                break
            raise
    train_losses.append(epoch_loss / max(1, len(X_tr) // batch_size_train))
    with torch.no_grad():
        mlp.eval()
        val_pred = mlp(X_val)
        val_losses.append(loss_fn(val_pred, y_val).item())
        mlp.train()

mlp.eval()
with torch.no_grad():
    y_pred_val = mlp(X_val).cpu().numpy()
y_val_np = y_val.cpu().numpy()
mae_mlp = mean_absolute_error(y_val_np, y_pred_val)
r2_mlp  = r2_score(y_val_np, y_pred_val)
print(f'MLP Latency Predictor:')
print(f'  MAE={mae_mlp:.2f}ms  R^2={r2_mlp:.4f}')

# MC-dropout uncertainty for 10 representative requests
X_sample = X_val[:10]
mean_pred, std_pred = mlp.predict_with_uncertainty(X_sample, n_samples=40)
print('\nMC-Dropout Uncertainty Estimates (10 requests):')
print(f'{"Request":>9} {"Mean(ms)":>10} {"Std(ms)":>10} {"p95_bound(ms)":>14}')
for i, (m, s) in enumerate(zip(mean_pred.cpu().numpy(), std_pred.cpu().numpy())):
    p95 = m + 1.645 * s
    print(f'{i:>9} {m:>10.1f} {s:>10.1f} {p95:>14.1f}')

# SLA admission at P75 (conservative allocation)
SLA_THRESHOLD = 200.0
p75_preds = mean_pred.cpu().numpy() + 0.674 * std_pred.cpu().numpy()
admitted = (p75_preds <= SLA_THRESHOLD).sum()
print(f'\nAdmitted at P75 (SLA={SLA_THRESHOLD}ms): {admitted}/10')

# Compare predictor quality: linear vs MLP
lr_val = LinearRegression().fit(X_all[:split], y_all[:split])
y_lr_val = lr_val.predict(X_all[split:])
mae_lr = mean_absolute_error(y_val_np, y_lr_val)
print(f'\nComparison:')
print(f'  Linear MAE:  {mae_lr:.2f}ms  R^2={r2_score(y_val_np, y_lr_val):.4f}')
print(f'  MLP MAE:     {mae_mlp:.2f}ms  R^2={r2_mlp:.4f}')
print(f'  MLP improvement: {(mae_lr - mae_mlp) / mae_lr:.1%}')

# Feature importance via ablation
feature_names = ['seq_len', 'batch_sz', 'model_B', 'fp16', 'kv_tokens']
base_mae = mae_mlp
print('\nFeature Ablation (importance by MAE increase when zeroed):')
with torch.no_grad():
    for fi, fname in enumerate(feature_names):
        X_ablated = X_val.clone()
        X_ablated[:, fi] = 0.0
        y_abl = mlp(X_ablated).cpu().numpy()
        delta_mae = mean_absolute_error(y_val_np, y_abl) - base_mae
        bar = '+' * int(delta_mae / 2)
        print(f'  {fname:<12}: +{delta_mae:.2f}ms MAE  {bar}')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(train_losses, label='Train', color='steelblue')
axes[0].plot(val_losses,   label='Val',   color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Huber Loss')
axes[0].set_title('Training Curves')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].scatter(y_val_np, y_pred_val, alpha=0.4, s=20, label='MLP')
axes[1].scatter(y_val_np, y_lr_val,   alpha=0.4, s=20, label='Linear', color='coral')
mn, mx = y_val_np.min(), y_val_np.max()
axes[1].plot([mn, mx], [mn, mx], 'k--', linewidth=1.5)
axes[1].set_xlabel('Actual Latency (ms)')
axes[1].set_ylabel('Predicted Latency (ms)')
axes[1].set_title('Predicted vs Actual: MLP vs Linear')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

axes[2].errorbar(
    range(10),
    mean_pred.cpu().numpy(),
    yerr=1.96 * std_pred.cpu().numpy(),
    fmt='o', capsize=4, capthick=1.5, color='steelblue', label='95% CI',
)
axes[2].axhline(SLA_THRESHOLD, color='red', linestyle='--', label=f'SLA {SLA_THRESHOLD}ms')
axes[2].set_xlabel('Request Index')
axes[2].set_ylabel('Predicted Latency (ms)')
axes[2].set_title('MC-Dropout Uncertainty Estimates')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Level 2 complete: MLP predictor with MC-dropout uncertainty')


## Real-World Example 1: Production Request Routing

In [ ]:
# Real-World Example 1: Online latency prediction with EMA updates
# Handle non-stationarity: latency distributions drift as traffic changes
# Use exponential moving average to continuously update predictor statistics

class OnlineLatencyPredictor:
    """
    Online latency predictor using EMA feature statistics.
    Adapts to distribution shift without full retraining.

    At each observed request:
    1. Update EMA of feature stats (mean, variance per feature)
    2. Standardize features with EMA stats
    3. Return prediction from lightweight linear model
    4. Update model weights via online gradient step
    """
    def __init__(
        self,
        n_features: int = 3,
        ema_alpha: float = 0.05,
        lr: float = 0.01,
    ):
        self.alpha = ema_alpha
        self.lr    = lr
        self.n     = n_features
        # Online feature statistics (EMA mean/variance)
        self.feat_mean = np.zeros(n_features)
        self.feat_var  = np.ones(n_features)
        # Linear model weights
        self.w = np.zeros(n_features)
        self.b = 50.0  # bias initialized to 50ms
        self.n_observed = 0

    def _update_stats(self, x: np.ndarray) -> None:
        """Update EMA feature mean and variance."""
        delta = x - self.feat_mean
        self.feat_mean += self.alpha * delta
        self.feat_var   = (1 - self.alpha) * (self.feat_var + self.alpha * delta ** 2)

    def _normalize(self, x: np.ndarray) -> np.ndarray:
        return (x - self.feat_mean) / (np.sqrt(self.feat_var) + 1e-9)

    def predict(self, x: np.ndarray) -> float:
        """Predict latency from raw features."""
        x_norm = self._normalize(x)
        return float(self.w @ x_norm + self.b)

    def update(self, x: np.ndarray, actual_latency: float) -> float:
        """Observe result, update model with SGD step, return prediction error."""
        self._update_stats(x)
        x_norm = self._normalize(x)
        pred = self.w @ x_norm + self.b
        error = actual_latency - pred
        # SGD gradient for MSE loss
        self.w += self.lr * error * x_norm
        self.b += self.lr * error * 0.1
        self.n_observed += 1
        return abs(error)

# Simulate two phases: stable phase -> distribution shift (traffic burst)
np.random.seed(42)
N_STABLE  = 200
N_SHIFTED = 200

# Phase 1: stable traffic (seq_len 64-256, batch 1-8, model_B 7)
X_stable = np.column_stack([
    np.random.randint(64, 256, N_STABLE).astype(float),
    np.random.choice([1, 2, 4, 8], N_STABLE).astype(float),
    np.ones(N_STABLE) * 7.0,
])
lat_stable = 10 + 0.05 * X_stable[:, 0] + 3.5 * X_stable[:, 1] + 5 * X_stable[:, 2]
lat_stable += np.random.normal(0, 8, N_STABLE)
lat_stable = np.maximum(lat_stable, 5.0)

# Phase 2: distribution shift – longer seqs, larger batches (traffic burst)
X_shifted = np.column_stack([
    np.random.randint(512, 2048, N_SHIFTED).astype(float),
    np.random.choice([16, 32, 64], N_SHIFTED).astype(float),
    np.ones(N_SHIFTED) * 70.0,  # larger model
])
lat_shifted = 10 + 0.05 * X_shifted[:, 0] + 3.5 * X_shifted[:, 1] + 5 * X_shifted[:, 2]
lat_shifted += np.random.normal(0, 15, N_SHIFTED)
lat_shifted = np.maximum(lat_shifted, 5.0)

X_all_online = np.vstack([X_stable, X_shifted])
y_all_online = np.concatenate([lat_stable, lat_shifted])

# Run online predictor
online_pred = OnlineLatencyPredictor(n_features=3, ema_alpha=0.08, lr=0.02)
# Comparison: static predictor trained on phase-1 only
static_reg = LinearRegression().fit(X_stable, lat_stable)

errors_online = []
errors_static = []
predictions_online = []

for i, (x, y) in enumerate(zip(X_all_online, y_all_online)):
    # Record predictions before update
    pred_online = online_pred.predict(x)
    pred_static = static_reg.predict([x])[0]
    predictions_online.append(pred_online)
    err_online = online_pred.update(x, y)
    errors_online.append(err_online)
    errors_static.append(abs(y - pred_static))

# Rolling MAE (window=20)
window = 20
rolling_online = [np.mean(errors_online[max(0, i-window):i+1])
                  for i in range(len(errors_online))]
rolling_static = [np.mean(errors_static[max(0, i-window):i+1])
                  for i in range(len(errors_static))]

# Summary statistics
mae_online_stable  = np.mean(errors_online[:N_STABLE])
mae_online_shifted = np.mean(errors_online[N_STABLE:])
mae_static_stable  = np.mean(errors_static[:N_STABLE])
mae_static_shifted = np.mean(errors_static[N_STABLE:])

print('Online vs Static Predictor: MAE Summary')
print(f'{"Predictor":<16} {"Phase 1 (stable)":>18} {"Phase 2 (shifted)":>19}')
print('-' * 56)
print(f'{"Static LR":<16} {mae_static_stable:>16.1f}ms {mae_static_shifted:>17.1f}ms')
print(f'{"Online EMA":<16} {mae_online_stable:>16.1f}ms {mae_online_shifted:>17.1f}ms')
print(f'Online adapts to shift: '
      f'{(mae_static_shifted - mae_online_shifted) / mae_static_shifted:.0%} MAE reduction')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rolling_online, label='Online (EMA)', color='steelblue', linewidth=2)
axes[0].plot(rolling_static, label='Static LR',    color='coral',     linewidth=2)
axes[0].axvline(N_STABLE, color='black', linestyle='--', linewidth=1.5,
                label=f'Distribution shift (t={N_STABLE})')
axes[0].set_xlabel('Request Index')
axes[0].set_ylabel('Rolling MAE (ms)')
axes[0].set_title('Online vs Static: Adaptation to Distribution Shift')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(y_all_online, alpha=0.5, linewidth=0.8, color='gray', label='Actual')
axes[1].plot(predictions_online, alpha=0.7, linewidth=1.2, color='steelblue', label='Online Pred')
axes[1].axvline(N_STABLE, color='red', linestyle='--', linewidth=1.5, label='Shift')
axes[1].set_xlabel('Request Index')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Actual vs Online Predictions Over Time')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Example 1 complete: online EMA predictor adapts to distribution shift')


## Real-World Example 2: SLA Compliance Monitoring

In [ ]:
# Real-World Example 2: SLA violation detection as binary classification
# Frame as: "will this request exceed 200ms p99?" -> binary label
# Use gradient-boosted trees (sklearn) vs logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, roc_curve,
)

np.random.seed(42)

# Generate dataset: classify requests as "will exceed 200ms SLA?"
def generate_sla_violation_dataset(n: int = 1000) -> tuple:
    seq_len    = np.random.randint(64, 4096, n).astype(float)
    batch_sz   = np.random.choice([1, 2, 4, 8, 16, 32, 64], n).astype(float)
    model_B    = np.random.choice([0.5, 3.0, 7.0, 13.0, 70.0], n)
    is_fp16    = np.random.randint(0, 2, n).astype(float)
    queue_depth = np.random.randint(0, 50, n).astype(float)  # requests waiting

    # True latency (ms)
    latency = (10
               + 0.06   * seq_len
               + 3.0    * batch_sz
               + 4.5    * model_B
               - 10.0   * is_fp16
               + 1.5    * queue_depth
               + 20.0   * np.random.randn(n))
    latency = np.maximum(latency, 5.0)

    # Binary label: 1 = SLA violation (>200ms), 0 = compliant
    labels = (latency > 200.0).astype(int)
    features = np.column_stack([seq_len, batch_sz, model_B, is_fp16, queue_depth])
    return features, labels, latency

X_sla, y_sla, lat_sla = generate_sla_violation_dataset(2000)
split = 1600
X_tr_sla, y_tr_sla = X_sla[:split], y_sla[:split]
X_te_sla, y_te_sla = X_sla[split:], y_sla[split:]

print(f'SLA violation dataset: {y_sla.sum()} violations out of {len(y_sla)} '
      f'({y_sla.mean():.1%})')
print(f'Train size: {len(X_tr_sla)}, Test size: {len(X_te_sla)}')

# Scale features
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_sla)
X_te_sc = scaler.transform(X_te_sla)

# Model 1: Logistic Regression (fast predictor, ~0.1ms inference)
lr_clf = LogisticRegression(C=1.0, max_iter=500, class_weight='balanced')
lr_clf.fit(X_tr_sc, y_tr_sla)
lr_proba = lr_clf.predict_proba(X_te_sc)[:, 1]
lr_pred  = (lr_proba >= 0.5).astype(int)

# Model 2: Gradient Boosted Trees (better accuracy, ~5ms inference)
gb_clf = GradientBoostingClassifier(n_estimators=80, max_depth=4, learning_rate=0.1)
gb_clf.fit(X_tr_sc, y_tr_sla)
gb_proba = gb_clf.predict_proba(X_te_sc)[:, 1]
gb_pred  = (gb_proba >= 0.5).astype(int)

print('\nClassification Report: Logistic Regression')
print(classification_report(y_te_sla, lr_pred, target_names=['Compliant', 'Violation'],
                             zero_division=0))
print('Classification Report: Gradient Boosted Trees')
print(classification_report(y_te_sla, gb_pred, target_names=['Compliant', 'Violation'],
                             zero_division=0))

# Summary comparison
print(f'{"Metric":<12} {"LogReg":>10} {"GBT":>10}')
print('-' * 35)
for metric, fn in [
    ('AUC-ROC', lambda p: roc_auc_score(y_te_sla, p)),
    ('Precision', lambda p: precision_score(y_te_sla, (p >= 0.5).astype(int), zero_division=0)),
    ('Recall',    lambda p: recall_score(y_te_sla,    (p >= 0.5).astype(int), zero_division=0)),
    ('F1',        lambda p: f1_score(y_te_sla,        (p >= 0.5).astype(int), zero_division=0)),
]:
    print(f'{metric:<12} {fn(lr_proba):>10.4f} {fn(gb_proba):>10.4f}')

# GBT feature importances
feat_names = ['seq_len', 'batch_sz', 'model_B', 'fp16', 'queue_depth']
print('\nGBT Feature Importances (SLA violation prediction):')
for name, imp in sorted(
    zip(feat_names, gb_clf.feature_importances_), key=lambda x: -x[1]
):
    bar = '|' * int(imp * 50)
    print(f'  {name:<12}: {imp:.4f}  {bar}')

# Admission control: use P75 calibration
# At P75 threshold (conservative): flag as "risky" if prob > 0.40
risk_threshold = 0.40
risky_lr = (lr_proba > risk_threshold).sum()
risky_gb = (gb_proba > risk_threshold).sum()
print(f'\nAdmission control (risk threshold={risk_threshold}):')
print(f'  LR flags {risky_lr}/{len(y_te_sla)} as risky')
print(f'  GBT flags {risky_gb}/{len(y_te_sla)} as risky')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
lr_fpr, lr_tpr, _ = roc_curve(y_te_sla, lr_proba)
gb_fpr, gb_tpr, _ = roc_curve(y_te_sla, gb_proba)
axes[0].plot(lr_fpr, lr_tpr, label=f'LogReg (AUC={roc_auc_score(y_te_sla, lr_proba):.3f})')
axes[0].plot(gb_fpr, gb_tpr, label=f'GBT    (AUC={roc_auc_score(y_te_sla, gb_proba):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[0].set_xlabel('FPR (False Alarm Rate)')
axes[0].set_ylabel('TPR (Detection Rate)')
axes[0].set_title('ROC Curve: SLA Violation Detection')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Feature importance
axes[1].barh(feat_names,
             gb_clf.feature_importances_,
             color='steelblue', alpha=0.8)
axes[1].set_xlabel('GBT Feature Importance')
axes[1].set_title('Top Drivers of SLA Violations')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()
print('Example 2 complete: SLA violation as binary classification')


## Real-World Example 3: SLA-aware Batching

In [ ]:
# Real-World Example 3: Sliding-window retraining feedback loop
# Retrain latency predictor on recent measurements every N requests
# Monitor model staleness and trigger retraining on drift detection

class SlidingWindowRetrainer:
    """
    Maintains a sliding window of (features, actual_latency) pairs.
    Retrains predictor every retrain_interval observations.
    Detects drift when validation MAE exceeds drift_threshold.
    """
    def __init__(
        self,
        window_size: int = 200,
        retrain_interval: int = 50,
        drift_threshold_ms: float = 20.0,
    ):
        self.window_size = window_size
        self.retrain_interval = retrain_interval
        self.drift_threshold = drift_threshold_ms
        self.buffer_X: list = []
        self.buffer_y: list = []
        self.model = Ridge(alpha=1.0)
        self.scaler = StandardScaler()
        self.n_observed = 0
        self.retrain_count = 0
        self.val_mae_log: list = []
        self.retrain_timestamps: list = []
        self.is_trained = False

    def observe(self, x: np.ndarray, actual_lat: float) -> Optional[float]:
        """
        Ingest one (features, latency) pair.
        Returns prediction error (None if not yet trained).
        """
        self.buffer_X.append(x)
        self.buffer_y.append(actual_lat)
        # Keep only window_size most recent
        if len(self.buffer_X) > self.window_size:
            self.buffer_X.pop(0)
            self.buffer_y.pop(0)
        self.n_observed += 1

        pred_error = None
        if self.is_trained:
            x_sc = self.scaler.transform([x])
            pred = self.model.predict(x_sc)[0]
            pred_error = abs(actual_lat - pred)

        # Trigger retraining every retrain_interval observations
        if (self.n_observed % self.retrain_interval == 0
                and len(self.buffer_X) >= 50):
            self._retrain()

        return pred_error

    def _retrain(self) -> None:
        """Retrain Ridge regression on current window."""
        X = np.array(self.buffer_X)
        y = np.array(self.buffer_y)
        # 80/20 train/val split within window
        split_idx = int(len(X) * 0.8)
        X_tr, y_tr = X[:split_idx], y[:split_idx]
        X_val, y_val = X[split_idx:], y[split_idx:]

        self.scaler.fit(X_tr)
        X_tr_sc  = self.scaler.transform(X_tr)
        X_val_sc = self.scaler.transform(X_val)
        self.model.fit(X_tr_sc, y_tr)
        val_preds = self.model.predict(X_val_sc)
        val_mae = mean_absolute_error(y_val, val_preds)

        self.val_mae_log.append(val_mae)
        self.retrain_timestamps.append(self.n_observed)
        self.retrain_count += 1
        self.is_trained = True

        drift = val_mae > self.drift_threshold
        if drift:
            print(f'  [t={self.n_observed}] Drift detected! val_MAE={val_mae:.1f}ms '
                  f'(threshold={self.drift_threshold}ms) -> retraining with window')

# Simulate 3 phases: stable, shift (burst), re-stable
np.random.seed(42)

def gen_requests(n, seq_range=(64, 512), batch_vals=(1, 4, 8), model_B=7.0):
    sl = np.random.randint(*seq_range, n).astype(float)
    bs = np.random.choice(batch_vals, n).astype(float)
    mb = np.full(n, model_B)
    lat = 10 + 0.05 * sl + 3.0 * bs + 4.0 * mb + np.random.normal(0, 10, n)
    X = np.column_stack([sl, bs, mb])
    return X, np.maximum(lat, 5.0)

# Phase 1: normal (0-300)
X1, y1 = gen_requests(300, seq_range=(64, 256), batch_vals=(1, 2, 4))
# Phase 2: traffic burst (300-500): longer seqs, larger batches
X2, y2 = gen_requests(200, seq_range=(512, 2048), batch_vals=(16, 32, 64))
# Phase 3: re-stabilize with 13B model (500-700)
X3, y3 = gen_requests(200, seq_range=(128, 512), batch_vals=(4, 8), model_B=13.0)

X_stream = np.vstack([X1, X2, X3])
y_stream = np.concatenate([y1, y2, y3])

retrainer = SlidingWindowRetrainer(
    window_size=200, retrain_interval=50, drift_threshold_ms=20.0
)
pred_errors: list = []
for x, y in zip(X_stream, y_stream):
    err = retrainer.observe(x, y)
    if err is not None:
        pred_errors.append(err)
    else:
        pred_errors.append(np.nan)

# Rolling MAE
ROLL_W = 30
rolling_mae = [
    np.nanmean(pred_errors[max(0, i - ROLL_W):i + 1])
    for i in range(len(pred_errors))
]

print(f'Sliding-Window Retrainer Summary:')
print(f'  Total observations:  {retrainer.n_observed}')
print(f'  Retraining cycles:   {retrainer.retrain_count}')
print(f'  Final window MAE:    {retrainer.val_mae_log[-1]:.2f}ms')
print(f'  Max window MAE:      {max(retrainer.val_mae_log):.2f}ms')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rolling_mae, linewidth=1.5, color='steelblue', label='Rolling MAE (w=30)')
for t in retrainer.retrain_timestamps:
    axes[0].axvline(t, color='green', alpha=0.3, linewidth=1.0)
axes[0].axvline(300, color='red', linestyle='--', linewidth=1.5, label='Burst start')
axes[0].axvline(500, color='orange', linestyle='--', linewidth=1.5, label='Re-stable')
axes[0].axhline(retrainer.drift_threshold, color='purple', linestyle=':',
                label=f'Drift threshold ({retrainer.drift_threshold}ms)')
axes[0].set_xlabel('Request Index')
axes[0].set_ylabel('Prediction MAE (ms)')
axes[0].set_title('Prediction Error: Sliding-Window Retrainer')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(retrainer.retrain_timestamps, retrainer.val_mae_log,
             marker='o', linewidth=2, color='coral', label='Val MAE at retrain')
axes[1].axhline(retrainer.drift_threshold, color='purple', linestyle=':',
                label=f'Drift threshold')
axes[1].set_xlabel('Observation Index (retrain trigger)')
axes[1].set_ylabel('Validation MAE (ms)')
axes[1].set_title('Model Quality Per Retraining Cycle')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Example 3 complete: sliding-window retraining with drift detection')


## Comparison: When to Use What

In [ ]:
# Comparison: Predictor architectures for latency SLA prediction
import matplotlib.pyplot as plt
import numpy as np

# Representative benchmark results
predictors = ['Linear\nRegression', 'Ridge\nRegression', 'Gradient\nBoosted\nTrees', 'MLP\n(3-layer)']
mae_ms       = [18.5, 17.2, 9.8, 7.4]    # Mean Absolute Error (ms)
r2_scores    = [0.82, 0.84, 0.93, 0.96]
pred_lat_ms  = [0.1,  0.1,  2.5,  1.8]   # predictor inference latency
train_sec    = [0.02, 0.02, 8.0,  12.0]  # training time (seconds)
online_update = [True, True, False, True] # supports online update?

# Simulated accuracy vs sparsity curve for each predictor
seq_lens_eval = np.linspace(64, 4096, 50)
true_lat = 10 + 0.06 * seq_lens_eval   # simplified 1-feature truth

preds_lr = 12 + 0.055 * seq_lens_eval + np.random.normal(0, 18, 50)
preds_ridge = 11 + 0.057 * seq_lens_eval + np.random.normal(0, 17, 50)
preds_gbt = 10 + 0.059 * seq_lens_eval + np.random.normal(0, 10, 50)
preds_mlp = 10 + 0.0595 * seq_lens_eval + np.random.normal(0, 8, 50)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

# 1. MAE comparison
axes[0, 0].bar(predictors, mae_ms, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('Mean Absolute Error (ms)')
axes[0, 0].set_title('Prediction MAE (lower is better)')
for i, v in enumerate(mae_ms):
    axes[0, 0].text(i, v + 0.3, f'{v:.1f}ms', ha='center', fontsize=10, weight='bold')
axes[0, 0].grid(alpha=0.3, axis='y')

# 2. R² score
axes[0, 1].bar(predictors, r2_scores, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('R² Score')
axes[0, 1].set_title('Explained Variance (R²)')
axes[0, 1].set_ylim([0.75, 1.0])
for i, v in enumerate(r2_scores):
    axes[0, 1].text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=10, weight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Predictor overhead
axes[0, 2].bar(predictors, pred_lat_ms, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[0, 2].set_ylabel('Inference Latency (ms)')
axes[0, 2].set_title('Predictor Overhead (must be < 5ms)')
for i, v in enumerate(pred_lat_ms):
    axes[0, 2].text(i, v + 0.05, f'{v:.1f}ms', ha='center', fontsize=10, weight='bold')
axes[0, 2].axhline(5.0, color='red', linestyle='--', label='5ms budget')
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3, axis='y')

# 4. Predicted vs actual scatter (MLP)
axes[1, 0].scatter(true_lat, preds_gbt, alpha=0.5, s=25, label='GBT', color='seagreen')
axes[1, 0].scatter(true_lat, preds_mlp, alpha=0.5, s=25, label='MLP', color='steelblue')
axes[1, 0].scatter(true_lat, preds_lr,  alpha=0.3, s=25, label='LR',  color='coral')
axes[1, 0].plot([true_lat.min(), true_lat.max()],
                [true_lat.min(), true_lat.max()], 'k--', linewidth=1.5)
axes[1, 0].set_xlabel('True Latency (ms)')
axes[1, 0].set_ylabel('Predicted Latency (ms)')
axes[1, 0].set_title('Prediction Quality: LR vs GBT vs MLP')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# 5. Training time
axes[1, 1].bar(predictors, train_sec, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('Training Time (s)')
axes[1, 1].set_title('Retraining Cost (for sliding-window retraining)')
for i, v in enumerate(train_sec):
    axes[1, 1].text(i, v + 0.2, f'{v:.2f}s', ha='center', fontsize=9, weight='bold')
axes[1, 1].grid(alpha=0.3, axis='y')

# 6. Selection guide
axes[1, 2].axis('off')
table_data = [
    ['Linear/Ridge',  '< 1ms overhead', 'Low accuracy', '< 50ms SLA budget'],
    ['GBT',           'Best accuracy',   'No online upd', 'Offline batch, high acc'],
    ['MLP',           'GPU-optional',    'Needs tuning',  'Balanced accuracy+speed'],
    ['Online EMA',    'Adapts to drift', 'Moderate acc',  'Distribution shift risk'],
]
tbl = axes[1, 2].table(
    cellText=table_data,
    colLabels=['Predictor', 'Pro', 'Con', 'Best for'],
    loc='center', cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.0, 1.6)
axes[1, 2].set_title('Predictor Selection Guide', weight='bold', pad=15)

plt.tight_layout()
plt.show()

print('Latency SLA Prediction: Method Comparison')
print('=' * 72)
print(f'{"Predictor":<20} {"MAE(ms)":>8} {"R2":>6} {"Overhead":>10} '
      f'{"TrainSec":>10} {"OnlineOK":>10}')
print('-' * 72)
for pred_name, mae, r2, oh, tr, onl in zip(
    predictors, mae_ms, r2_scores, pred_lat_ms, train_sec, online_update
):
    pred_str = pred_name.replace('\n', ' ')
    print(f'{pred_str:<20} {mae:>8.1f} {r2:>6.3f} {oh:>9.1f}ms {tr:>9.2f}s '
          f'{"Yes" if onl else "No":>10}')
print('\nKey insight: GBT or MLP + online EMA is the recommended production setup')
print('Key insight: predictor overhead must stay <5ms to not eat into SLA budget')


## Key Takeaways

**Core idea:** Predicting request latency before execution enables SLA-aware routing, batching, and scheduling decisions that maintain quality-of-service guarantees.

### Prediction Model Comparison

| Model | Accuracy | Inference Time | When to Use |
|-------|----------|----------------|-----------| 
| Linear | ~82% | <1ms | Low-latency production |
| Neural Net | ~88% | 1-2ms | **Default recommendation** |
| Ensemble | ~91% | 5-10ms | High-accuracy requirements |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| SLA violations spike after deployment | Training data distribution mismatch | Retrain on recent production data |
| Predictions always pessimistic | Model overly conservative | Recalibrate threshold or confidence margin |
| Memory errors despite predictions | Complex feature engineering forgotten | Simplify features or cache preprocessed values |

## Exercises

1. **Train a multi-feature predictor:** Add features like batch_size, model_size, and cache_hit_rate. Does accuracy improve?

2. **Online learning:** Implement incremental SLA model updates as actual latencies arrive. Does it adapt faster?

3. **SLA SLO tracking:** Monitor prediction error and actual SLA compliance rates. When does the model drift?